<a href="https://colab.research.google.com/github/NoT-Serna/Juan_Serna_FlyRank/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

From the toolkit, I will use logistic regression because my target variable, "needs_review", is a binary classification label with two possbile outcomes True or False

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

The split design will be time-aware, this will include a training dataset being the selection of the month of April. May and June will be used as a validation period. This prevents information form future loads form entering the training data and better reflects the outputs of the ML model

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
import numpy as np
import pandas as pd
import duckdb
from google.colab import userdata
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Read the token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# Create DuckDB connection
con = duckdb.connect()

# Register the Hugging Face secret
con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

# Warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test the connection
con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""").show()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘



**Week 4 - Baseline**

In [6]:
# ============================================================
# 1. APRIL DATA — FEATURES + WEEK-4 BASELINE
# ============================================================

april = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month = '2025-04'
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# CTR
april["ctr"] = np.where(
    april["impressions"] > 0,
    april["clicks"] / april["impressions"],
    0
)

# Position bucket
april["position_bucket"] = pd.cut(
    april["avg_position"],
    bins=[0, 3, 5, 10, 20, float("inf")],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

# Expected CTR from Week-4 signal analysis
expected_ctr = {
    "1-3": 0.019535,
    "3-5": 0.018284,
    "5-10": 0.008456,
    "10-20": 0.006893,
    "20+": 0.002238
}

april["expected_ctr"] = (
    april["position_bucket"]
    .astype(str)
    .map(expected_ctr)
)

april["ctr_gap"] = (
    april["expected_ctr"] - april["ctr"]
)

# Week-4 baseline prediction
april["baseline_prediction"] = (
    (april["ctr_gap"] > 0) &
    (april["impressions"] >= 100)
).astype(int)


# ============================================================
# 2. MAY + JUNE FUTURE OUTCOME
# ============================================================

future = con.sql(f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    AVG(gsc_avg_position) AS avg_position
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
WHERE month IN ('2025-05', '2025-06')
GROUP BY
    content_hash_id
""").df()

# Future CTR
future["ctr"] = np.where(
    future["impressions"] > 0,
    future["clicks"] / future["impressions"],
    0
)

# Future position bucket
future["position_bucket"] = pd.cut(
    future["avg_position"],
    bins=[0, 3, 5, 10, 20, float("inf")],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

future["expected_ctr"] = (
    future["position_bucket"]
    .astype(str)
    .map(expected_ctr)
)

future["ctr_gap"] = (
    future["expected_ctr"] - future["ctr"]
)

# Future ground-truth outcome
future["future_needs_review"] = (
    (future["ctr_gap"] > 0) &
    (future["impressions"] >= 100)
).astype(int)

# ============================================================
# 3. JOIN APRIL FEATURES WITH FUTURE OUTCOME
# ============================================================

data = april.merge(
    future[["content_hash_id", "future_needs_review"]],
    on="content_hash_id",
    how="inner"
)

print("Rows available for evaluation:", len(data))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows available for evaluation: 12725


**Logistic Regression on April**

In [8]:
features = [
    "impressions",
    "avg_position",
    "ctr"
]

X = data[features]
y = data["future_needs_review"]

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(max_iter=1000))
])

model.fit(X,y)

data["logistic_prediction"] = model.predict(X)

f1 = f1_score(
    y,
    data["logistic_prediction"]
)

print("Logistic Regression F1:", f1)

Logistic Regression F1: 0.7736937873667793


The Logistic Regression model achieved an F1 score of 0.77 indicating a somewhat strong balance between precision and recall when identifying pages with the label "needs_review".

The target was derived from the Week-4 Baseline rule, this result measures how well the model reproduces the existing baseline decicion rather than its ability to predict idependent future SEO outcomes

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The Logistic Regression is useful as a baseline, but its performance suggests that the problem may contain nonlinear relationships that logistic regression cannot capture. Therefore deciding the "needs_review" has more complex relationships that cannot be fully captured by the logistic regression.

## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.